In [1]:
from main_pipeline import (
    load_data,
    temporal_split,
    get_feature_groups,
    get_final_model,
    evaluate_model
)


In [7]:
# ==========================================
# SHARED PIPELINE IMPORTS (DO NOT MODIFY)
# ==========================================

from main_pipeline import load_data, temporal_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# Load dataset
df = load_data("master_dataset_enhanced.csv")

# Temporal split (shared logic)
train, test = temporal_split(df)

# Define X and y
X_train = train.drop(columns=["yield"])
X_test = test.drop(columns=["yield"])

y_train = train["yield"]
y_test = test["yield"]

# -------------------------------------------------
# Remove temporal lag features (Phase 2 integrity)
# -------------------------------------------------

lag_cols = [
    col for col in X_train.columns
    if col.startswith("yield_lag")
    or col.startswith("yield_ma")
    or col.startswith("yield_change")
]

X_train = X_train.drop(columns=lag_cols)
X_test = X_test.drop(columns=lag_cols)

print("✓ Shared pipeline loaded")
print("Removed lag features:", lag_cols)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

✓ Shared pipeline loaded
Removed lag features: ['yield_lag1', 'yield_ma2', 'yield_change']
Train shape: (17510, 34)
Test shape: (2179, 34)


In [8]:
print("\n" + "="*70)
print("🔬 ABLATION STUDY")
print("="*70)

# -------------------------------
# Feature Groups
# -------------------------------

group1 = ['N', 'P', 'K', 'pH',
          'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent']

group2 = ['NPK_total',
          'N_to_P_ratio', 'N_to_K_ratio', 'P_to_K_ratio',
          'NPK_balance_score']

group3 = ['temp_rainfall_interaction',
          'temp_humidity_interaction',
          'moisture_index',
          'growing_degree_days']

group4 = ['pH_optimal', 'soil_fertility_score']

group5 = ['fertilizer_per_ha',
          'pesticide_per_ha',
          'fertilizer_rainfall_ratio',
          'pesticide_efficiency',
          'input_intensity']

categorical_cols = ['crop', 'state', 'season']

experiments = {
    'Baseline (Base)': group1,
    '+ NPK Ratios': group1 + group2,
    '+ Climate': group1 + group2 + group3,
    '+ Soil Quality': group1 + group2 + group3 + group4,
    '+ Input Efficiency': group1 + group2 + group3 + group4 + group5
}

results_ablation = []

for name, feature_set in experiments.items():

    selected_features = feature_set + categorical_cols
    
    X_train_sub = X_train[selected_features]
    X_test_sub = X_test[selected_features]
    
    # Dynamically define feature types
    numerical_cols = X_train_sub.select_dtypes(include=['int64','float64']).columns.tolist()
    categorical_cols_sub = X_train_sub.select_dtypes(include=['object']).columns.tolist()
    
    # Build fresh preprocessor
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    preprocessor_sub = ColumnTransformer([
        ('num', numeric_pipeline, numerical_cols),
        ('cat', categorical_pipeline, categorical_cols_sub)
    ])
    
    model = Pipeline([
        ('preprocessing', preprocessor_sub),
        ('model', ExtraTreesRegressor(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    model.fit(X_train_sub, y_train)
    y_pred = model.predict(X_test_sub)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results_ablation.append({
        'Configuration': name,
        'Feature_Count': len(selected_features),
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

ablation_df = pd.DataFrame(results_ablation)

baseline_mae = ablation_df.iloc[0]['MAE']
ablation_df['Δ_MAE_from_Baseline_%'] = (
    (baseline_mae - ablation_df['MAE']) / baseline_mae
) * 100

print("\n📊 Ablation Results:")
print(ablation_df)

ablation_df.to_csv('ablation_study_results.csv', index=False)
print("\n✓ Saved: ablation_study_results.csv")



🔬 ABLATION STUDY

📊 Ablation Results:
        Configuration  Feature_Count       MAE      RMSE        R2  \
0     Baseline (Base)             10  1.197834  3.898743  0.928940   
1        + NPK Ratios             15  1.190726  3.897148  0.928998   
2           + Climate             19  1.200691  3.945167  0.927237   
3      + Soil Quality             21  1.191159  3.941342  0.927378   
4  + Input Efficiency             26  1.097110  3.714340  0.935503   

   Δ_MAE_from_Baseline_%  
0               0.000000  
1               0.593367  
2              -0.238512  
3               0.557231  
4               8.408821  

✓ Saved: ablation_study_results.csv


In [5]:
pd.read_csv("ablation_study_results.csv")


,Configuration,Feature_Count,MAE,RMSE,R2,Δ_MAE_from_Baseline_%
0,Baseline (Base),10,1.197834,3.898743,0.928940,0.000000
1,+ NPK Ratios,15,1.190726,3.897148,0.928998,0.593367
2,+ Climate,19,1.200691,3.945167,0.927237,-0.238512
3,+ Soil Quality,21,1.191159,3.941342,0.927378,0.557231
4,+ Input Efficiency,26,1.097110,3.714340,0.935503,8.408821


In [6]:
[col for col in X_train.columns if "lag" in col or "ma" in col or "change" in col]

['pH_optimal', 'yield_lag1', 'yield_ma2', 'yield_change']

In [ ]:

plt.figure(figsize=(9,6))

plt.plot(ablation_df['Configuration'],
         ablation_df['R2'],
         marker='o',
         linewidth=2)

plt.xticks(rotation=45)
plt.ylabel('R²')
plt.title('Ablation Study – Incremental Feature Contribution')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_study_plot.png', dpi=300)
plt.show()


In [ ]:
print("\n" + "="*70)
print("🔍 FEATURE IMPORTANCE – FINAL ABLATION MODEL")
print("="*70)

# Rebuild full feature set
full_features = group1 + group2 + group3 + group4 + group5 + ['crop','state','season']

X_train_full = X_train[full_features]

# Dynamically define feature types
numerical_cols = X_train_full.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols_sub = X_train_full.select_dtypes(include=['object']).columns.tolist()

# Build preprocessing pipeline
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_full = ColumnTransformer([
    ('num', numeric_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols_sub)
])

final_model = Pipeline([
    ('preprocessing', preprocessor_full),
    ('model', ExtraTreesRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X_train_full, y_train)

# Extract feature names after preprocessing
preprocessor_fitted = final_model.named_steps['preprocessing']
model_fitted = final_model.named_steps['model']

# Numerical feature names
num_features = numerical_cols

# Encoded categorical names
cat_features = preprocessor_fitted.named_transformers_['cat'] \
    .named_steps['encoder'] \
    .get_feature_names_out(categorical_cols_sub)

all_features = list(num_features) + list(cat_features)

importances = model_fitted.feature_importances_

importance_df = pd.DataFrame({
    'feature': all_features,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 15 Important Features:")
print(importance_df.head(15))


In [ ]:
print("\n" + "="*70)
print("📊 STATISTICAL SIGNIFICANCE TEST")
print("="*70)

# ---------------------------
# Helper function to build pipeline
# ---------------------------

def build_pipeline(feature_list):
    
    X_train_sub = X_train[feature_list]
    
    num_cols = X_train_sub.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_cols = X_train_sub.select_dtypes(include=['object']).columns.tolist()
    
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, num_cols),
        ('cat', categorical_pipeline, cat_cols)
    ])
    
    model = Pipeline([
        ('preprocessing', preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    return model

# ---------------------------
# Baseline Model
# ---------------------------

baseline_features = group1 + ['crop','state','season']
baseline_model = build_pipeline(baseline_features)

baseline_model.fit(X_train[baseline_features], y_train)
baseline_pred = baseline_model.predict(X_test[baseline_features])

# ---------------------------
# Final Model
# ---------------------------

full_features = group1 + group2 + group3 + group4 + group5 + ['crop','state','season']
final_model = build_pipeline(full_features)

final_model.fit(X_train[full_features], y_train)
final_pred = final_model.predict(X_test[full_features])

# ---------------------------
# Bootstrap Significance Test
# ---------------------------

n_bootstrap = 1000
baseline_scores = []
final_scores = []

np.random.seed(42)

for _ in range(n_bootstrap):
    indices = np.random.choice(len(y_test), len(y_test), replace=True)
    
    baseline_scores.append(
        r2_score(y_test.iloc[indices], baseline_pred[indices])
    )
    
    final_scores.append(
        r2_score(y_test.iloc[indices], final_pred[indices])
    )

t_stat, p_value = stats.ttest_rel(final_scores, baseline_scores)

print("\nPaired t-test Results:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
